In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dimitrinovareze/dictionnaire/dico.csv
/kaggle/input/datasets/dimitrinovareze/moliere/moliere.txt


In [6]:
import numpy as np
# joli affichage du modèle *m*
def summary(m):
    total_p = 0
    for i, p in enumerate(m.parameters()):
        num_p = np.prod(p.shape)
        if p.requires_grad:
            trainable = "entrainables"
        else:
            trainable = "figés"
        print(f"Couche {i}: {list(p.shape)} (\x1b[34m{num_p}\x1b[0m paramètres {trainable})")
        total_p += num_p
    print(f"  = \x1b[31m{total_p}\x1b[0m paramètres entrainables")



def summary_perso(m):
    total_p = 0
    for i, p in enumerate(m.parameters()):
        num_p = np.prod(p.shape)
        if p.requires_grad:
            trainable = "entrainables"
        else:
            trainable = "figés"
        total_p += num_p
    return total_p

In [28]:
# Cours sur la compréhension d'un décodeur de type GPT
# MSO 3ème année de l'option Information
# Ecole Centrale de Lyon
# Julien VELCIN

# Ce script correspondant à une implémentation complète d'un décodeur de type GPT
# Il est directement inspiré de la vidéo "Let's build GPT: from scratch, in code, spelled out" d'A. Karpathy (Director of AI at Tesla, OpenAI)
# https://www.youtube.com/watch?v=kCc8FmEb1nY
# Voir aussi son implémentation appelée NanoGPT : https://github.com/karpathy/nanoGPT (sous licence MIT)

# on rajoute ici :
# - normalisation de couche (layer normalization)
# - connections résiduelles (residual connections)
# - plusieurs blocs Transformer (stack of Transformer blocks)

# chargement des données

import torch
import torch.nn as nn
import torch.nn.functional as F
import re

#hyperparameters



###################
batch_size = 32  # nombre de séquences traitées en parallèle
block_size = 64  # context length: combien de caractères allons-nous regarder pour prédire
max_iters = 5000
eval_interval = 300
learning_rate = 1e-3
eval_iters = 200
n_embd = 256  # taille des embeddings
n_block = 6  # Ajout de l'hyperparametre du nombre de block et du nombre de tete d'attention
n_head = 8
#####################





# accélération à l'aide d'un GPU (ou pas...)
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

torch.manual_seed(1337)

# chargement des données
with open("/kaggle/input/datasets/dimitrinovareze/moliere/moliere.txt", "r", encoding="utf-8") as file:
    text = file.read()



def clean_corpus(raw_text):
    cleaned = re.sub(r' {2,}', ' ', raw_text)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    # Supprime les espaces restants en tout début ou toute fin de ligne
    cleaned = re.sub(r'(?m)^ +| +$', '', cleaned)
    cleaned = cleaned.replace('\t', ' ')
    return cleaned
    
text = clean_corpus(text)



# construction du vocabulaire (ici, les caractères uniques dans le texte), qu'on appelle un codebook
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}   
encode = lambda s: [stoi[c] for c in s] # encoder: str -> list of int
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: list of int -> str

# encodage du dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))  # 90% pour le train
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # génère un batch de données
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad() # pas de backpropagation dans cette fonction, on ne fait qu'évaluer
def estimate_loss():
    out = {}
    model.eval() # mode évaluation
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = model(X, y)
            losses[k] = loss.item()
        avg_loss = losses.mean()
        out[split] = avg_loss
        out[f"{split}_ppl"] = torch.exp(torch.tensor(avg_loss)).item()    #On ajoute la perplexité ici !
    model.train() # mode entraînement (ici, ne sert à rien car on n'a pas de dropout ou batchnorm)
    return out

# ajout d'une classe pour gérer la self attention
class Head(nn.Module):
    # gère une seule tête d'attention
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # register_buffer permet de stocker un tensor qui n'est pas un paramètre entraînable 
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C), avec C = head_size
        q = self.query(x) # (B,T,C), avec C = head_size
        v = self.value(x) # (B,T,C), avec C = head_size
        # calcule le score d'attention (attention maps)
        wei = q @ k.transpose(-2,-1) * C**-0.5  # (B,T,T), **-0.5 = racine carrée
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf')) # (B,T,T)
        # attention, ici on est obligé de "couper" la matrice pour correspondre à la taille de la séquence
        # car celle-ci peut être plus petite que block_size lors de la génération
        wei = torch.softmax(wei, dim=-1)
        out = wei @ v  # (B,T,T) @ (B,T,C) -->  (B,T,C)
        return out

# classe pour gérer plusieurs têtes d'attention
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out
    
# deux couches feedforward exécutées pour chaque token indépendamment
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

# classe pour un bloc Transformer
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # version 1 :
        #x = self.sa(x) 
        #x = self.ffwd(x)
        # version 2 = avec connections résiduelles / skip connections
        #x = x + self.sa(x)
        #x = x + self.ffwd(x)
        # version 3 = avec connections résiduelles *et* layer normalization
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        # à noter que l'ordre exact des opérations d'addition, normalisation...
        # n'est pas exactement le même que celui du Transformer original,
        # mais c'est une variante qui fonctionne très bien et plus simple à implémenter
        return x

class myGPT(nn.Module):
    def __init__(self):
        super().__init__()
        # les embeddings (lookup table) :
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # plus un embedding positionnel :
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # ici on modifie pour avoir de l'attention multi-têtes
        self.blocks = nn.Sequential(
            *[Block(n_embd,n_head) for _ in range(n_block)],
            nn.LayerNorm(n_embd)
        )
        # ajout d'une couche linéaire pour projeter les embeddings vers le vocabulaire
        self.lm_head = nn.Linear(n_embd, vocab_size)  

    def forward(self, idx, targets=None):
        B, T = idx.shape # on récupère les dimensions du batch et de la séquence
        # idx et targets sont des tensors de forme (B, T) où B est le batch size et T la séquence length
        tok_emb = self.token_embedding_table(idx)  # (B, T, C) où C est la dimension des embeddings (n_embd)
        # ajout des embeddings positionnels
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, C)
        x = tok_emb + pos_emb  # (B, T, C)
        # appliquer les blocs Transformer définis dans self.blocks
        x = self.blocks(x)  # (B, T, C)
        logits = self.lm_head(x)  # (B, T, vocab_size) 

        if targets is None: # nécessaire pour la génération de texte (càd sans vérité terrain)
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # à expliquer (illustration)
            targets = targets.view(B*T) # à expliquer (illustration)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx est un tensor de forme (B, T)
        for _ in range(max_new_tokens):
            # couper le contexte pour avoir maximum block_size tokens
            idx_cond = idx[:, -block_size:]  # (B, block_size) 
            # on demande la prédiction du prochain token
            logits, _ = self(idx_cond) # (B, T, C)
            logits = logits[:, -1, :]  # prendre les logits du dernier token (B, C), -1 sélectionne le dernier élément de la 2ème dimension (le temps)
            probs = F.softmax(logits, dim=-1)  # (B, C), on calcule la proba sur la dernière dimension (les classes)
            next_idx = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, next_idx), dim=1)  # concaténer le nouveau token à la fin de la séquence
        return idx





    ###################### BEAM SEARCH ####################################
    def generate_BEAM(self, idx, max_new_token, k, pr=False):
        idx = idx.repeat(k, 1) # (k, T)
        scores = torch.zeros(k, device=device)
        scores[1:] = -1e9 #pour pas générer que des espaces
        for _ in range(max_new_token):
            # on demande la prédiction du prochain token
            logits, _ = self(idx[:, -block_size:]) # (B, T, C)
            logits = logits[:, -1, :] # prendre les logits du dernier token (B, C), -1 sélectionne le dernier élément de la 2ème dimension (le temps)

            #on passe en log pour pas avoir des valeur trop petite en multipliant les probs
            log_probs = F.log_softmax(logits, dim=-1)  # (B, C), on calcule la proba sur la dernière dimension (les classes)
            
            scores = scores.unsqueeze(1) + log_probs  #log prob de taille (k,V) et score.unsqueeze(1)


            flat_scores = scores.view(-1)
            best_score, best_idx = torch.topk(flat_scores,k)

            #best_indices contient des nombres entre 0 et (k×V)−1
            pistes_meres = best_idx // vocab_size
            lettres_choisi = best_idx % vocab_size 
           
          

            idx = idx[pistes_meres]
            nouvelle_colonne = lettres_choisi.unsqueeze(1)
            idx = torch.cat((idx, nouvelle_colonne), dim=1)


            scores = best_score

        meilleur_index = torch.argmax(scores)
        return idx[meilleur_index].unsqueeze(0)




    ##############################BEAM SEARCH AVEC PENALITE ##############################
    def genrate_BEAM_avec_penalite(self, idx, max_new_token, k):
        idx = idx.repeat(k, 1) # (k, T)
        scores = torch.zeros(k, device=device)
        for _ in range(max_new_token):
            # on demande la prédiction du prochain token
            logits, _ = self(idx[:, -block_size:]) # (B, T, C)
            logits = logits[:, -1, :] # prendre les logits du dernier token (B, C), -1 sélectionne le dernier élément de la 2ème dimension (le temps)
            
            
            repetition_penalty = 1.5 
            
            for i in range(k):
                recent_tokens = torch.unique(idx[i, -50:]) #On pénalise sur les 50 derniers tokens
                
                for token in recent_tokens:
                    if logits[i, token] > 0:
                        logits[i, token] /= repetition_penalty
                    else:
                        logits[i, token] *= repetition_penalty
                        
                        
            log_probs = F.log_softmax(logits, dim=-1)  # (B, C), on calcule la proba sur la dernière dimension (les classes)
            
            scores = scores.unsqueeze(1) + log_probs  


            flat_scores = scores.view(-1) #1D
            best_score, best_idx = torch.topk(flat_scores,k)

            #On aa applatit les scores, il faut retrouver les indices et les lettres de notre piste qui a le meilleur résultat
            pistes_meres = best_idx // vocab_size
            lettres_choisi = best_idx % vocab_size 
           
          

            idx = idx[pistes_meres]
            nouvelle_colonne = lettres_choisi.unsqueeze(1)
            idx = torch.cat((idx, nouvelle_colonne), dim=1)


            scores = best_score

        meilleur_index = torch.argmax(scores)
        return idx[meilleur_index].unsqueeze(0)
        

    


### Quelques metrique (perplexité calculé dans estimate_loss)

In [8]:
## Tester sur les lettres et avoir une entropy sur la dispertions des tokens 
def calculate_lexical_diversity_word(text):
    words = text.lower().split()
    if not words: return 0
    return len(words)/len(set(words))


In [9]:
from collections import Counter

#Va nous permettre d'afficher l'histogramme de la répartitions des token dans le texte.
def calculate_lexical_diversity_token(text):
    return Counter(text)

In [10]:
def build_reference_vocab(raw_text):
    mots = re.findall(r'\b[a-zàâçéèêëîïôûùüÿæœ]+\b', raw_text.lower())
    return set(mots) 

MOLIERE_VOCAB = build_reference_vocab(text) 


def calculate_hallucination_rate(generated_text, reference_vocab):
    mots_generes = re.findall(r'\b[a-zàâçéèêëîïôûùüÿæœ]+\b', generated_text.lower())
    
    if len(mots_generes) == 0:
        return 0.0, []
        
    mots_inventes = []
    for mot in mots_generes:
        if mot not in reference_vocab:
            mots_inventes.append(mot)
        
    taux = len(mots_inventes) / len(mots_generes)
    
    return taux, mots_inventes

## Passage à l'echelle 

In [12]:


batch_size = 64       
block_size = 128      
max_iters = 6000    
eval_interval = 1000 
eval_iters = 200      

learning_rate = 3e-4 #plus lent car plus gros modele

n_embd = 256         
n_head = 8           
n_block = 6          

import time

print("Lancement du Maxi Modèle Molière...")
model = myGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

debut = time.time()

for iter in range(max_iters):
    # Évaluation périodique
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Iter {iter} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
        
    # Apprentissage
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

duree = (time.time() - debut) / 60
print(f"Entraînement terminé en {duree:.2f} minutes.")

torch.save(model.state_dict(), "modele_moliere_MAXI_1.pt")
print("Modèle sauvegardé sous 'modele_moliere_MAXI_1.pt'")





🚀 Lancement du Maxi Modèle Molière...


/tmp/ipykernel_55/4125984740.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out[f"{split}_ppl"] = torch.exp(torch.tensor(avg_loss)).item()    #On ajoute la perplexité ici !


Iter 0 | Train Loss: 4.8869 | Val Loss: 4.8933
Iter 1000 | Train Loss: 1.5844 | Val Loss: 1.6712
Iter 2000 | Train Loss: 1.3098 | Val Loss: 1.4571
Iter 3000 | Train Loss: 1.1865 | Val Loss: 1.3763
Iter 4000 | Train Loss: 1.1123 | Val Loss: 1.3495
Iter 5000 | Train Loss: 1.0551 | Val Loss: 1.3409
Iter 5999 | Train Loss: 0.9994 | Val Loss: 1.3498
Entraînement terminé en 17.39 minutes.
💾 Modèle sauvegardé sous 'modele_moliere_MAXI_25k.pt'


##### 

In [13]:
summary(model)

Couche 0: [114, 256] (29184 paramètres entrainables)
Couche 1: [128, 256] (32768 paramètres entrainables)
Couche 2: [32, 256] (8192 paramètres entrainables)
Couche 3: [32, 256] (8192 paramètres entrainables)
Couche 4: [32, 256] (8192 paramètres entrainables)
Couche 5: [32, 256] (8192 paramètres entrainables)
Couche 6: [32, 256] (8192 paramètres entrainables)
Couche 7: [32, 256] (8192 paramètres entrainables)
Couche 8: [32, 256] (8192 paramètres entrainables)
Couche 9: [32, 256] (8192 paramètres entrainables)
Couche 10: [32, 256] (8192 paramètres entrainables)
Couche 11: [32, 256] (8192 paramètres entrainables)
Couche 12: [32, 256] (8192 paramètres entrainables)
Couche 13: [32, 256] (8192 paramètres entrainables)
Couche 14: [32, 256] (8192 paramètres entrainables)
Couche 15: [32, 256] (8192 paramètres entrainables)
Couche 16: [32, 256] (8192 paramètres entrainables)
Couche 17: [32, 256] (8192 paramètres entrainables)
Couche 18: [32, 256] (8192 paramètres entrainables)
Couche 19: [32, 25

Tentative de Beam Search

In [34]:
gc.collect()
torch.cuda.empty_cache()
model.eval()

# sans le torch no grad on avait une erreure CUDO out of memory (surtout pour le beam search)
with torch.no_grad():
    generated_idx = model.generate(context, max_new_tokens=500)

text = decode(generated_idx[0].tolist())

print("--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---")
print(text)

print(f"----------------------- \nLa diversité lexicale du texte est : {calculate_lexical_diversity_word(text)}")

--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---
Mais pour qu'un jeune époux effet de vos royaumes,
Veneuse, qui satisferoit à heux qu'elle paye,
Ouvre jouer le rôle au lui devait d'Agnès la difficulté.
Nous aurons changements du _pris_, genardère aux cabuts
souvent elle comique que sa consolette ainsi.

[114] N'avez-vous pas dit, du grand Sganarelle? Le voilà qui m'est soufflé
de trouver d'affaire.

CLITANDRE.

Quand signe obligée[147]?

SGANARELLE.

D'où vous vient? C'est, que personne le ciel ne se détourne;
Et sertoit au meilleur de l'inéga
----------------------- 
La diversité lexicale du texte est : 1.1142857142857143


In [35]:
calculate_hallucination_rate(text,MOLIERE_VOCAB)

(0.10465116279069768,
 ['royaumes',
  'veneuse',
  'satisferoit',
  'heux',
  'genardère',
  'cabuts',
  'consolette',
  'sertoit',
  'inéga'])

In [30]:
gc.collect()
torch.cuda.empty_cache()
model.eval()

# sans le torch no grad on avait une erreure CUDO out of memory
with torch.no_grad():
    generated_idx = model.generate_BEAM(context, max_new_token=500, k=3)

text = decode(generated_idx[0].tolist())

print("--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---")
print(text)

print(f"----------------------- \nLa diversité lexicale du texte est : {calculate_lexical_diversity_word(text)}")

--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---
Monsieur, je vous demande pardon de tout ce que je vous dois.

MOLIÈRE.

Mais, monsieur, vous vous moquez, monsieur, que je vous prie.

DON JUAN.

Madame, je vous prie.

SGANARELLE, à part.

Oui, monsieur, je vous prie.

Il faut que j'aille une femme qui me considère.

SGANARELLE, à part.

Je vous demande pardon que je ne suis pas comme vous voulez.

SGANARELLE, à part.

Allons, monsieur, je vous prie, et je ne suis pas comme vous.

DON JUAN.

Monsieur, vous vous moquez.

SGANARELLE.

Monsieur, j
----------------------- 
La diversité lexicale du texte est : 2.073170731707317


In [31]:
calculate_hallucination_rate(text,MOLIERE_VOCAB)

(0.0, [])

In [32]:
gc.collect()
torch.cuda.empty_cache()
model.eval()

# sans le torch no grad on avait une erreure CUDO out of memory
with torch.no_grad():
    generated_idx = model.genrate_BEAM_avec_penalite(context, max_new_token=500, k=3)

text = decode(generated_idx[0].tolist())

print("--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---")
print(text)

print(f"----------------------- \nLa diversité lexicale du texte est : {calculate_lexical_diversity_word(text)}")

--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---
Monsieur, j'avois
déchaîné.--Tout là; mais je perçais que je fasse, et voudrois
chez l'état.) Je ne m'y puis rien faire, je voudrois
que chez-vous?...

DONE IGNÈS.

Mais l'objet, monsieur; que chérit de vous père!..

ARNOLPHE.

Mais, monsieur?

GÉRONTE.

J'étois là-dessus; je vous parle, monsieur?

SGANARELLE.

Où d'honnêtes-je fâché! Je vais, monsieur?

DAPHNÉ.

Ell'est de plaisir à chérir que je ne m'oblige;
Mais, enfin, dès que votre âme échappe.
Je voulois bien à m'exposer, et je dois que cha
----------------------- 
La diversité lexicale du texte est : 1.4181818181818182


In [33]:
calculate_hallucination_rate(text,MOLIERE_VOCAB)

(0.03296703296703297, ['perçais', 'ell', 'cha'])

### Dans la partie précedente, nous avions "clean" le corpus, en enlevant les espaces, je propose ici de réentrainer le meme modèle sans ces espaces pour effectuer une comparaison

In [36]:
with open("/kaggle/input/datasets/dimitrinovareze/moliere/moliere.txt", "r", encoding="utf-8") as file:
    text = file.read()

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))  # 90% pour le train
train_data = data[:n]
val_data = data[n:]

In [37]:


batch_size = 64       
block_size = 128      
max_iters = 6000    
eval_interval = 1000  
eval_iters = 200      

learning_rate = 3e-4  
n_embd = 256          
n_head = 8            
n_block = 6         


import time

model = myGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

debut = time.time()

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Iter {iter} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
        

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

duree = (time.time() - debut) / 60
print(f"Entraînement terminé en {duree:.2f} minutes.")

torch.save(model.state_dict(), "modele_moliere_MAXI.pt")
print("Modèle sauvegardé sous 'modele_moliere_MAXI.pt'")





/tmp/ipykernel_55/506668507.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  out[f"{split}_ppl"] = torch.exp(torch.tensor(avg_loss)).item()    #On ajoute la perplexité ici !


Iter 0 | Train Loss: 4.9388 | Val Loss: 4.9641
Iter 1000 | Train Loss: 1.5176 | Val Loss: 1.5792
Iter 2000 | Train Loss: 1.2552 | Val Loss: 1.3839
Iter 3000 | Train Loss: 1.1438 | Val Loss: 1.3093
Iter 4000 | Train Loss: 1.0706 | Val Loss: 1.2605
Iter 5000 | Train Loss: 1.0176 | Val Loss: 1.2582
Iter 5999 | Train Loss: 0.9714 | Val Loss: 1.2634
Entraînement terminé en 17.62 minutes.
Modèle sauvegardé sous 'modele_moliere_MAXI.pt'


In [ ]:
summary(m)

In [41]:
gc.collect()
torch.cuda.empty_cache()
model.eval()

# sans le torch no grad on avait une erreure CUDO out of memory (surtout pour le beam search)
with torch.no_grad():
    generated_idx = model.generate(context, max_new_tokens=500)

text = decode(generated_idx[0].tolist())

print("--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---")
print(text)

print(f"----------------------- \nLa diversité lexicale du texte est : {calculate_lexical_diversity_word(text)}")

--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---
MAIRE.

  «Non, monsieur, que vous puis-je être entrée, mon fils?

  ALAIN.

  Oui.

  CÉLIE.

          Tranc, après un bille petit proisonné,

  LUCILE.

                                    Paix..

  ASCAGNE.

                                               Monsieur..

  TRUFFACALDIN.

  MASCARILLE, à part.

  L'espérance m'était où sert dans ce tour.

  LÉLIE.

  Son frère: ne te rapporoit pas victoire.

  MARINETTE.

                                  Tais-tu?

  AGNÈS.

                       
----------------------- 
La diversité lexicale du texte est : 1.0


In [42]:
calculate_hallucination_rate(text,MOLIERE_VOCAB)

(0.12244897959183673,
 ['maire', 'tranc', 'bille', 'proisonné', 'truffacaldin', 'rapporoit'])

Sur la méthode de generation de base, le text généré sans clean du corpus est moins pollué par les numéros des pages qu'on aurait du enlever. Le résultat est alors bien différent de ce qu'on a obtenu précdemment

In [39]:
gc.collect()
torch.cuda.empty_cache()
model.eval()

# sans le torch no grad on avait une erreure CUDO out of memory
with torch.no_grad():
    generated_idx = model.generate_BEAM(context, max_new_token=500, k=3)

text = decode(generated_idx[0].tolist())

print("--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---")
print(text)

print(f"----------------------- \nLa diversité lexicale du texte est : {calculate_lexical_diversity_word(text)}")

--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---
MASCARILLE.

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        
----------------------- 
La diversité lexicale du texte est : 1.0


Comme expliqué dans le rapport, BEAM se contente d'écrire des espaces.

In [44]:
gc.collect()
torch.cuda.empty_cache()
model.eval()

# sans le torch no grad on avait une erreure CUDO out of memory
with torch.no_grad():
    generated_idx = model.genrate_BEAM_avec_penalite(context, max_new_token=500, k=3)

text = decode(generated_idx[0].tolist())

print("--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---")
print(text)

print(f"----------------------- \nLa diversité lexicale du texte est : {calculate_lexical_diversity_word(text)}")

--- TEXTE GÉNÉRÉ VIA BEAM SEARCH ---
Mais cette profére,
  Et qu'à même devoir le bâton dans cette profére,
  Que j'y prenois...

  ALCESTE.

                              Monsieur, j'attendois.

  PHILÈNE.

  Oh! comme voulez-vous, j'ai brêté?

  ARSINOÉ.

  Quoi! chère quel moyen, j'ai des petits;
  Et vous...

  ALBERT.

               Malgré ce qu'il vous pût, dès là;
  Enfin.--LÉANDRE, MASCARILLE.
  PHILÈNE, courante-là! j'y présends; mais, que vous
  QUATRE.

  MYNTILÈS.

  Oh! c'étoit-on dans le même, et jugerai bien que
  Ly
----------------------- 
La diversité lexicale du texte est : 1.2352941176470589


La pénalité est suffisante pour enlevé le problème de ne générer que des espaces même si le résultat semble moins satisfaisant.

In [45]:
calculate_hallucination_rate(text,MOLIERE_VOCAB)

(0.08, ['profére', 'profére', 'brêté', 'présends', 'myntilès', 'ly'])

Augmentation du taux d'hallucination par rapport à précédemment